# A Deep Network from Scratch: Data, Code, Training, Results, Pitfalls
### CSCI 4220 · in-class coding · *UDL ch. 4 — Deep neural networks* (previews ch. 5–8)

We build and train a deep network with nothing but numpy, and use it to learn a picture of
Robby's cat from pixel positions alone. Every line that does machine learning, you write.

$$\text{data} \;\rightarrow\; \texttt{init\_params} \;\rightarrow\; \underbrace{\texttt{forward} \rightarrow \texttt{loss} \rightarrow \texttt{backward} \rightarrow \text{update}}_{\text{repeat}}$$

| min | | you write |
|---|---|---|
| 0–9 | **1** · the data | `X`, `Y` |
| 9–18 | **2** · the network | `relu`, `init_params`, `forward` |
| 18–26 | **3** · loss and backpropagation | `loss`, `backward`, a gradient check |
| 26–36 | **4** · training, and reading the result | `train_gd`, Adam |
| 36–44 | **5** · does it generalize? | a train/test split |
| 44–54 | **6** · common pitfalls | five fixes |
| 54–60 | **7** · the mystery painting | your best network |

**Blanks:** type the commented line directly above each `______`.

▶ **Run these two cells once** — imports, then the drawing helpers (no machine learning in them).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── drawing helpers: run once, never edit (every line of machine learning is in the cells below) ──
import base64, io
from types import SimpleNamespace
from matplotlib import animation
from matplotlib.patches import Rectangle
from IPython.display import HTML
from PIL import Image, ImageDraw

np.seterr(all="ignore")                         # we detect NaN ourselves instead of printing warnings
INK, GRAY, FAINT, BLUE, RED = "#0b0b0b", "#52514e", "#b9b8b3", "#2a78d6", "#c62828"
plt.rcParams.update({"axes.grid": True, "grid.alpha": .18, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11, "axes.titleweight": "bold", "axes.titlelocation": "left"})
_SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")

def robbys_cat(n=48):
    """Robby's cat, drawn big, then shrunk to n x n pixels. Values 0-1, float32."""
    S = 480; im = Image.new("RGB", (S, S), (214, 234, 248)); d = ImageDraw.Draw(im)
    d.rectangle([0, 360, S, S], fill=(236, 214, 170))
    for s in (1, -1):
        e = lambda x: 240 + s * (x - 240)
        d.polygon([(e(95), 175), (e(150), 55), (e(215), 140)], fill=(232, 140, 52))
        d.polygon([(e(120), 160), (e(152), 88), (e(195), 140)], fill=(246, 186, 190))
    d.ellipse([70, 110, 410, 420], fill=(232, 140, 52))
    for x0 in (205, 232, 259):
        d.rounded_rectangle([x0, 120, x0 + 14, 185], 7, fill=(176, 88, 30))
    d.ellipse([160, 280, 320, 400], fill=(252, 236, 214))
    for cx in (170, 310):
        d.ellipse([cx - 38, 205, cx + 38, 275], fill=(122, 196, 70))
        d.ellipse([cx - 9, 210, cx + 9, 270], fill=(20, 20, 20))
        d.ellipse([cx + 8, 220, cx + 18, 232], fill=(255, 255, 255))
    d.polygon([(222, 292), (258, 292), (240, 314)], fill=(232, 110, 140))
    d.line([(240, 314), (240, 332)], fill=(60, 40, 30), width=5)
    d.arc([200, 312, 242, 352], 0, 160, fill=(60, 40, 30), width=5)
    d.arc([238, 312, 280, 352], 20, 180, fill=(60, 40, 30), width=5)
    for s in (-1, 1):
        for dy in (-10, 12):
            d.line([(240 + s * 70, 322), (240 + s * 175, 322 + dy * 2)], fill=(250, 250, 250), width=4)
    d.rounded_rectangle([372, 300, 452, 410], 10, fill=(200, 45, 50))
    d.arc([430, 325, 478, 385], -90, 90, fill=(200, 45, 50), width=12)
    d.ellipse([376, 292, 448, 312], fill=(90, 55, 30))
    return np.asarray(im.resize((n, n), Image.LANCZOS), dtype=np.float32) / 255

def pixel_grid(n):
    """The same three lines you write in section 1, for an n x n grid."""
    coords = np.linspace(-1, 1, n, dtype=np.float32); x1, x2 = np.meshgrid(coords, -coords)
    return np.vstack([x1.ravel(), x2.ravel()])

_MYSTERY = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAABHGlDQ1BJQ0MgUHJvZmlsZQAAeJxjYGDiyUnOLWYSYGDIzSspCnJ3UoiIjFJgv8PAyCDJwMygyWCZmFxc4BgQ4MOAE3y7BlQNBJd1QWbhVocVcKWkFicD6T9AHJdcUFTCwMAYA2Rzl5cUgNgZQLZIUjaYXQNiFwEdCGRPALHTIewlYDUQ9g6wmpAgZyD7DJDtkI7ETkJiQ+0FAeZkIxJdTQQoSa0oAdFuTgwMoDCFiCLCCiHGLAbExgwMTEsQYvmLGBgsvgLFJyDEkmYyMGxvZWCQuIUQU1nAwMDfwsCw7XxyaVEZ1GopID7NeJI5mXUSRzb3NwF70UBpE8WPmhOMJKwnubEGlse+zS6oYu3cOKtmTeb+2suHXxr8/w8A3kFTfazGM+sAAB5ySURBVHjaJdrZsuXJeR32b8rM/7D3ObvOqbG7QYAEAZCmKCtMDZYvFKGQQheyHJZeyI/kF/CVb0SRlGlRhDmAIIEG0F3dVafOtPd/yMxv8EU/w4pYEesXC//P/+PfPb/nqrpzvj6My3j86LEbOeNpW4Zty2pmJrVheK+99F6YEAAwUCQxFSFGUjMA6KlgERgKIJVxcGFCE9gEemdukYADotYeABks2Nw5UcqBQAjMJBQAmihQF9adQX2P+oQ95Z0k0ugDtYdFN0GSfV8lfB3G7vvA4pzoGss1BeRJgXE8dbhZlxWEu2sgduTe9nAtYaweBJuhkqfWlAKZcyIiMDJCxaRJzVpvGruI557TM0dd+lzjSCQ8FGTJBClUrGc3ULCuYRaM5ughigDaOfd8gmnKDxci1q11oIGFQ00srmutAZZzO6XHoE+bD4LjWLI2crUoEYFBkQS9exM3jwBxhhBK3CdaB8IGAcDgYR2gkgbRcw/bAEwOmCaHnCpOC7zpPI2khYIpU7gsZzlvVDXU+rqBdfAIQqQgCErJhI1JH6C07dWVV5GPWxDpOBYzl1TG3rthsZo7MnPN3nOSIpKFFBolNKLeHcjNoG1KjIh7YmQiNDcrz1u1biQBge4u2MaidALIeZcbpLFC2nomBZaYcRs9eDdbzvq8r+viZh4OTOEOLC5EROTGGtjClobCQOjPHDvCZJFKX3vfu+RBzv9wqe/PI7bB3FNqyIFRAxYhC6chQ2EYMowlhBwjSFhKcwdACLRwJJZiEitjd0AqWWW6cDE5Kd9gyBB9sn3wjUD90nKFer60fW+9MyCWhNNMaIwYgIQIhB6uHmkkgHCHFcBb572lNurxaqufOMDDHVz2hwVvxr4DsEASBNCupBaMYIS1xWYQCMiOERBckh2KzNkzQ5ZIRWXe8daPxZAiEBCJqFAcRKWe+fzAahrSd90uNZal1hYMOI7lcEiAKADQkE2EaEyGGiDWhKyyGjqmm2Nuvj1YJdz32h7PppGIwSxUZUi8tWZ7gxwUHkQdQkQAwCkcJRCBMSxMDbXTeePzngaWkThzHlMqRMQk2cHMk3twVGq9N9fdWovaIRDNDdSFcTrMUogSATWSSqUFIUsuUhNt0d0DsWRBRc9d5q0cqTNAhjogRMqpt3VIFBDUVGBvUaTN83DZYNeURNiZg8lNMNAh2IBdwkQ8CkRBD7WgswH0TlYJmRHpEhEGZKYWgCAWwASGQIiJKEviGTh7cO9Ug22cbMrimsxJfLdd106XPu2Gm8re097zpfOifUo2Qg/GMfFRjBncg4VzIgHJPA749CBzGV4P83gJBODRIBEThgZQW3dki2i6eai7SyhKNVByDQjCFkAQwUDMaQjtQsQYAsBCaWZPWHUNvmB2B4JgiPm8De/v4nGDi/rFxqeeurOzoFCQEGALuHTZeh737crXK+6vp3i/7genq0KAKCXL8UejP11gLuP1AWp9rPm5UbVkig54HCdJHJqLdeHggalwQQUwsAretQJChLE5FI4s6gHmyORIpOHIK3Nz8N4F5dRavqzw6aL3Tb6t8mwCDEBsiMREKRiAiDyCAvaOEDgmR4cnkHPQ+03U2h/wWo6JEPVpkeHmUtvybXvz5a/r81M/4/ik3o07ojoOOQr0Qj5AFMKB7Ur0SDFJjEMZ5oEymjBGxLpRqdWrOUcSZgpHc2AqF50uNT/t/LjGY40l0kpDjwzZ0HoCU8g9QJgZzQMgYCgUwY0wLNADAFnylJ3csIx+2QHRzWVMouPvfFnvHzf5pPkycTMy2KeRMrFFIEUArmr3NnZD9Ci9T+ija0GYyCeBSVwAADNKCj86cjD1AANWhxXSorSbtYAg6ZIKhpia7+Dt8nQeSknZrMHeJYlgKgidWYgiD1IVazcJSond4UWJD09rRSYCROwIssPNY1sK7K+vh/MzOuZAfrDwjuQK3swpEJmCIBiBJFWSNULdwRWb66IJ40VOGjAyJsYAdyALXC1WNfdwDw5DsoCqGAFAgRFxGOaUZBgzD1KbDfOYJJmzBVjAspCDFSFJk3Y9Dijsx1y8XQxAhDFC/uqOv1wTtI5M41QSylZxRsyJxOrdY61tQZYMflUkCTMRoXY3EWSSbv5EbmYpFULczGoAUPZwcgjgzEEUTYNYCDElFGJtO4YzYSLcNYZpRANiyjlbb9q2HpTyMGR4bnya+DTnbz7ZHtTqqq2WAAf0AIwQYBzH1KJcHw6RWKt+dmUdh46kUCYpqapFdEhPJIcUjCpIBqLIQ0mCcOsqqVggQAxtmVjVQh0didGJqSpeNj1dj3PBtlcI7lYhuMnhFS+PHz7cr3tVR4QkgICnkQLloWNGJkR12mtclZineL7IOKQSPOSMCA4s//IHcBPDf/ul/Q+f8Zj60xpQ+2m2p0iTb3EyNQQMYrnsHm4YYZhYijqsTqhNUgpiQCvMZfZxzGhhrT21OO8GSKM4FO7bfndxbSqcEWMUPtj61LrK9G5OVRUQe0BJxFIGIdns49lvRznN6WGx0PaDibYds3Nf3QkKSxxH+cVXT3/y88v5fPnLdnl7O/3sfGKQf6O/Os2Abh7MICkxY5sHNQMzJ8TMtsV078fDSEF8f+5loJm1BW/7s1n6cCHFJLY/XPDScKkNXSPMQAIwUFpdKKK3hVincpSSpyyBoWoWEYG1h3kqOX14bPtumeMXHzEIse1IEAQeXrcuf/4XX316juLbs1x/+tbvn79BTn86//gLXHW/NyyI4JoZzoVdqDAH8ihCFoJKRmHqwVKV2GvYWlKv6eVFyjHz8hR3z0+OQoCIAcSmsSmk6QA0u2vvra3VXK/z6I6qrTeeppySQJaZYqm2d8oF9orbBY5w0egkjITkFERyLD4yHkY5Jr0ATQXX7enxMb0pSim9O5UhUVe0NibMFmRtA6tuKbEVxHDIQhl79H4QU7FGwy1/yiNempSrMg0HRNqrW2UMm4csg/zp3z10vxVs3nti8rrslxivj+E+jpISK5BHdA0iOU6pqxI494tCv+fQZjVoIiQhcYTAAGCoS4tGdBxKytSGgjPpKZNRBjemtLTk+SAjsGRBgoiAqG0hyrV5yrwKgHez2tUT08sDE0YbB8DA2N3lrh6eKviyMg/zPPTLSro6gYcgJbcgZpVp2fOyW7jOAx9nwehWL2IVeZYDP989ZCAzVwarXcBCm+cpF1fYzaEhguPwFIfq69PHbfAH4hAwYE/122mchzS4e63QzMQakZRchFmBLz1Qu21r5APIDWjvbRng3t1/+Th9fX82mS/9ajxM/fnT8vhVYDjQNI5C8byuw/EUFSI0CR/GMUG09UlbdQhJ0/e/d/X+8dsGmBADACLK1SxzEsH9edl/8G54/FiftsqMB3g+2iVnajRtkQ0zRDSn8EgYox3guznadgJFQzfpFk2jdeToQx4Zwi4foW9QLw9KH/TdN+f56qqIr0EuoCEpMAJJWIjFQMrhUA43bVsClKFn08tlC/DgMs/TD97OLPHmeP3VN88BAEyMeFk3KSXNQ//2XJ9WvT6VL8/1tnB/fvwmZsj5Up8DxaGWEgYZkQMd4BMQEuiECpTMc/i219rNnjbkVBL0MFQDIQl8qTgK86uT9P5k5iLDsq2pDON82J8fEMEDxukqZ1mffqNmBoyh+6oseT4eXp/GH72Wbe8bHMfsgtAjIsIR05iEwSa2wzAua4eRAmFT++0X6f7x7HkWxLtKkGeqJcoVRpeUALC13SLMlHQTQlUnYOSJArxjlCspeRQJVG0XCQWCdX9kZgO4LI/T1UkAuqQdKcIRmWFv26IEUIbElJGmfPViLvNIBYEjnlaDYstST4fh4bIHRAAIiQCisLx+Mb2h7a8/XIoMHP54bm9fHr76uB5flNc3/R+eah/mwjXaDgqSD3kaIIwoqxZVL+hIEMAAjhToBrEuy6LWmcWCam9A5HtTi2Ec23qpda2tAnNJZToWS51yPuQho2CCnOTNIFcZz92nnD481gg28y8/7FcjnxETUwAAgMyHHPexK7x8l2/WfP9ofBhzpruHZSrymw/bm5P9/ov4+3Wo/Gqagb322HrMiHFeLhg93NwCpRA6kXXV8AAAZsA0VeOmNVgE1MKH+ajbtpwfmaK3Pk/z4XpGiTHPqTAYdKBS8CT4uNtzpc9eQEL/zRZvr+GbDS7LLlwFISMC4TCPgm6ADoFbp3IY7e45zMabF3Z5+vbucnO6+rj1x8vT737hD/3+fX2dhH27V7xwuakGoJYkDM1s/66PjZJTMnXUPXGdJE5DZwSFYYFo7UKEw3GwvV0f5jxlT3Gdh/m6+GodkRGOBNbdAl9f+zTKhztPGTSgNmPypQYBEHgAbVuTm1cvru+/hb4Ow7R8Ckl57Xp+Xn7/89PjYq77m6vpV4/4F79cf/QKf1Lqz9pbHN8MvnV9OI1T16xmCuQWhEjEEOGqXpdTsh/e6HEitI7umuP+cvjVp/V570h0PBwS+Rrba5nmWQaPJRxywdoeVhyyfHaKacRlzwomHPerkKu2lgQionZLCevWCB3DfG2w9+XFVCmV79+Wutbtw/27o5zXHnV9fZTpMP/th/jqQ/tdeD/2b3colObl8nB+fNguZ2wtg2fO4i1bn2AR399ew3GIEjqwpQwp9ndXz99/WQikMInrbuvNYc4FwsKtKQOqDSyJ8WaSQDlv9HRRJry7QE7UjYFzxgDEAEQNEOb/+L/85Nf356WlKcGQ6LGl331VCvPP3z8xBA/T06Uy6NWYsQwPq9+t/m7ckn5o/GaYDxg7BCaBYczgFd2QPJaHMk2f3fA00Cn5IECUiCQRCMa6oyRy03KdE0yErfCwqX1a/eUsE/s4ppdzXKo9blES7z3KSBTwuMFcvNWVmX9yPWQkQKDW+suJXh3gstM4pFdHWR3e3aTr6+EfHqoIwXz1dGn7sl4nvznNTvJXH5wD3trf5n45HN+ejlfsjo5mNowF2+Xd68Prm3Kd1HsEp5Qm4hGhhElYezUZmqY5DzKyBXCunRb1cZDbEYcUL0ZnCiFwh4Hs1RUMyc+b/s7rOI2CLIBhZu7Wl0rg0joc2IGyAPzwVl/Mee37F29nzIPvizo+wrB28NivRsvDUER+9sHvLvYZ/fqlfSMyTi/eel8JIOqaheZDAe9qQZyelc4ozREjIkCdRRtRcOZBPaXGSBGGQnNhju4W2w5No5o7QS5AEMseU07HmQ2Rhc0cIxjR1CmNaTXezcYMj6u2qq7VaNrd3r48fPnsN7QRyb3K+4/96fH8arbDPF1dzV8/w0+/Cekfvo8/K4jl6vWB+iyROCUkhABkcEVKDhQWvVkwlMRACIxsdiWR0EdE80ZIE0fb27JZqHa1ZjSOkgs0hSFBkkCgwlSrQyAjQABlIbVgSYYhCAqpiOxVE1GglIRShq/P+sXku/sF0uPFPz0+H6c6ZZnnwS1++rWvtf+Y/vvcvql0XBVROAuQW6Q8TLnXui5dA1B6BSBKnBEAyAOFECgCHICZELwDisRxcgfsDokNAgFQkmNYt3h5NeUEIEBAasaJSCOse4vs2gBQGIoghA154iQ3M+0h75f2gzFWxcjjsvG3d9uruX5xSocsAPCnv9a/uUu/d/j2e/mrvfpSobU2UD9vITkL+LYDAtTQwHl7vKBkc4CwgLTu/WFtQiEEFG6BIh4Re4MgGpG0B5FDAKE0A2E4TcURwtXdvRkhBHhIkpTQUc6rZ8Jda2FgSimngeyp0UXtJ1fwYVUosu/wN79aJuoD2zQNx8P8dx/2P/11eT3HP3336UDLN/fx6lTscnncAIXr3p92S9P1fq5J4OICIkyh3aqZKjhRFuFARWRCINwVCkaEqhMTGiALRiABZEYE0kA3mK5HYqR5wIRRTYprxWKBYf7peZmSBRERgbevN+Ah/9aEH8+dh+IKP/3VRc3eHPjqeLi5eXH33P7kF7y18Y8+30786dLyi4N/fVeN5wa8a7j2VrXm4fGsCZ0AAY0xIjqTQWDHFOATBhPVFplBINRh04CwnNmAhO3ti/FqSglBCBCAkDkLSuAeZF0p3EkG4cvWa/UpY8opEYDDX3/aXs78dsS7tQ2HCTw+na2wvztslOjN69ut63/9tf3y0/HNKe/rxsPV49OyLDuiM3gEaOCm0bqaB3q4WkJhYcFI5B1w767ElxrmQYIOUl3cgZAsINwzWWYqTEJI4WlIxMyAmNAIfY+M7ltDUxgy7a0fOU5zvh7nuVDb+pdn+/FNOSV+qF4O41jo5x838fbDwy5CN7fXAv5Q6c++TO8/VjDoFndnfX1NpzGalrpXClTVlDgzYIAgM0lrRKHI2FwQZe/gHhmhGQfA9SgsGAEApA6tq5mRkLl764SIkmQesiAqggdB9EtP6mgoGHg60otDeX01vDiUT5f21Wb/40thgl2x5FJ3/avfNAS6Satb3J4O562bmfa+VF13u3+ujzvVSu+/vdyvnQjBAbtmwbA9E0xk2syqC2AAujoFMIWAZ0HVtgMiECM4sgcZEEcQgEOECFFi4mRhBgHsWyV2A8IsiQg2GEby2xcyZXp3k8chf/1cn5T+pxcCpsvero+TO//ll5dA/Pwkz2vvnBxwFgTAXhtEjOActlS/P++t9ZJwX9e9Wu0toCGroHdtVut3apaJmCAQODSzm4V1IIiwjuBCQMySeciybzvlkgEJIRixByl47WDWnQZwf9itKqekx0M+jOndiwTBf/vYUpIf36TOEoC3p4Mg/v1Xq4eCUOveas0JTTWjfro0SXKOdHfp67Y9PC2CQATL3prb5XJJYYcMbtatEcLmYEDI2RzMICgQQBUQAwMjyM2/Y3MkgHWnUDUIDUzuHgBC6GSmvbagFN4XGNAkwgbh11fDaZZW21886RdDelOgdhfh3/rsFQJ+dVdFyMODKCWuywKIZv5Xv1l/9vVaVVMZvv64fLh7viZ/M221eQT22g+JMoV2JY5FtTtwzluj7qHOQQ4oa1U1BcSubm7o4I6cB/4P/+ofn3dzt8R42SOYRwRH7A4kxayjlIMQQXeEwuQe507rtl0cB22NBVlKknlOtemymzONEBb0zblXDyLuDq321i3CwW3MZdnbP3nTr0rdQZ43C6Ii9B3fu3oRnsexWWRQJyGCQUCQeuBVDgb79ly/YCS1vRtBHry7MA+JEyEAAnoCBLKwYOQwW50ZqRt0x2FI11NKnN4/rp8avkm9pHhu2hzmeYIItJCUeo9t93Bwh8Iw5RyO4RAB3eCi+MtHuUnw+7eX11e6butaGzEi2VFcKKLtI0Hv0Q0pcBI/lMiCg8SUsXl0t4DotRFE9ED1IEIkDA/DkKDEhaJJzhzK6Mw8CnUPFr6dZZ5KEO7up1evf1jUEYUEWSIAiCFlSEMSdkByl4AADDAAN8DwAKCff9BffBhozb9/Y7/7qtW2Pl5Wc8dwRhOoYD0VGlC7mSrkhPOYmcAxMlMJcA9OQmvtrRshAOFxSgiAKRFoloThQswpDajoQAA5lyTpapLbMUkZLlv//+53Kscf2vO2d1XSACZm4ao7U0TwMQmn4toSZ0ICImQS5gj+6wf/4/f4dE+/nfwfvXPx/fywmGm3DhSSPIC8LWuzHhARBKGOZlibtqWROxESoRCn78IdGAoAEAK5EFMqAt1xXA3dFDllBvNgluuZrqcZhvH+qw//z0O/HeS3JvxwroR4yESIvYPkLCxjZjUD5CFhgAMiCY1jBkog8qny//2b+Puv8m2V//kdvj60x6cFqvYIJ1LrCBZ1x2hr1bqua7Wh5MQIZoaoe6ckKCIG4AHuRkKm4JJyW8ZxGL0eCi6dw2FXiOjEbAaHMd1OMY0Tlvzxw6ef+nSE0L5FykOR1sM8AikRQkpAhBBCFICEgigeCCyqbh5E8McP+sdfQzzAP7kdf/sNL8tT3RoboLZOPIm1vfVuzUKIIACDvblpKAQNBRkjM4VHQABhV2/EGB0Dp5IOfbureFaKbXvacyZ3TGj4ds6vZuJpItXffHj6pWUEnIoszfa6E9i+Lpk8iNE9U2TBCEBAlrTudSrpxWHUXp92m4h+sej/dS93dfj+1fjjV37krerm23JwxfAIknAC7+Z777upIbCwIpFgqKsFmUMiQgQIXCp4YmprHw7h9TQdPmzEDNqCSJih0yCCx1GuxjGPQ13Wrz89BAB5PN4/We8QTgClpFqVXHMSDQoP1bZsW2stM5ehnI6HQfBu1RGx7v5fPtLfP6aBDl9cp++lb8Tu1/M51Z3CmvYcHcMRk6rtAEhgXQkBsyAQdDOEEHRi2Lo7kcQOSC7xItk4jpvRxFu3NApsSOeO81BK4mGepeRt72G6Nm3WBB3Dx8wApBGEEUDq4BEIGKaH+cpUwTtLPkzjzSHdb03dcat/+fX+3x/KXvME6Ye3cjM+crFWN1UPq4Re3bqFelh4YSCRkgjdohs2tTEhICHSoixJsFXOJdvzy5E3pyHJeV1aFLA90ojgr2fJzGk6QJLMsq+XRGjaI1wQg3AgEwRHXutGgEOZjuPwwxflj74YpjHPYwZgjXh7lbvHc4sS+v7T5S+/afeXI+zH7x+P70ZLeF63iweG2XdoSQRlEFDg//Cv/6ht/WlrewuhKImagRpqwJyJvHMaBCqirAbAbBEYOKTojsQyk0JwA6mqvdYId0SWBGbzOKKk64QWsBpc1m0WpPCnpWbqn9P+6sXh8Pbd+fmcy0SJrxJc9rZ2fDH4stf3LWWkiXAwux6jmj4texjklD88br8jNI5pWSr/r//qn6177+ZVFcLmIltXRqgKOdGAjWlwoTA1LM11SLRVm8cJvM5Xtxk0k1WNAERtTTWlzIgZ7XQ67c2PEpdmm9m+bt97fXNgNY2zyyrTYXu8yaFj1m4lFR6GkaFpXZofkoT232xYgQ8QM8XtAUpua9PLOXazz0xLzm7O/9u//qcP59jrThhddS7cDSlA1ZyxCIqGyxCxMg/PFa9K7IoDaUrTtq+UB9YNAD0EGbYeJXHr/WrMZRhVexG4GHUzBHj78pqYeq+ttY8taJpP/fFltD4dtmY5FUkyczS1p+oQPgquzVa1A9LAZcx2O7cWG7V06z4cSljwf/q3//zh04UZPEC7TpkDSRWRsBteTextZxkDHcGWXhD2lERVM4eGMAkL1dYM+LL31jVxuMFhHKuCJCQRc1uqjkPOjGYgwq4VVT/uhsN4oPW27TFOq0ZhFhZ0016roUNIX/eu3zTi7kcczh3evqWbQfRTTLcHA+f//d/+i/PTJRE4kntkxpxZGwDj1uMwZtRKAY6iXolSICYONRqx5fFYVZEzunbKl733WkchZByYNSIREqGqX7aWBTygMOSUEMN6B4tvNodydSvxBjYtw8MeySMzEJN5qNlmkMLB+i+eNSiOMjpNy93uj3U4jcTfcbiUfW+ZgpjUghkdo6kT4aVaOHbrwodjkZsRpzILI+YELBIrU3pcF2VZW3NigHBANVdzsAjTdW9b7Rj9zXXa3S8WBDBOhzyMheAQ+vO7y9cX8oftJ/D0o9elYoyux1KYeGAmwkel2j2F/fnXy3/+8v7prtYHq1VBgxkpMJJga12EswiShDkSVHVEurTQNHlYR0S+Tr6gGUYS6NXctFmYAArzINwtgCkJU0Q30IAWqN23Wqckt9w+v+Ktx9otYZzm8TjngX0O+2+L/goO7dP6A3z4w99+0UvOtiPAPORXxyKAdzsk4c+P+WHFP/nZ3bL1zWNduvWgJJwTA5I65MxVY9uUESjA1Wr3Spxztv35uacWFH19asQATkxEM/VjIXA9XV+zZAjwcARQNWGcshBEV8eUGdO4nd+c0qetL83mxNM0D8NQoI1W/8ul/b823311np8/fv759QU5QseMn5/KZy/GnPibxdTi1YGlpBbBwmFGItS3fZhyAKoGElhYUzPrRcDVKKJ3DyTa18RUoyD087KfW2bC5oGugByhrW/DMLj73g0QFQwRIGw1D6bV4AyFRQ59eXWav11saXqYhpvrw+lQZtbB+0/P+mfLeH7/9PLp7g9/60VOpL2D4fdej//y916lnL+8aBA1s5xTEsYilJj//b/4g61569qaz2MKgyHLYUwA3lq4kwdcTQm1p5wNuFlNgYhO4UycEZt5zly1fzr7vizfHTGyMBN3971rV/cAQr49pBdsx0KXSPdbvLouV8fMnLdt1aY59GOPlcpbjNuDN8y69MyUEn/v5XA15V+d6979euBrxGJ2uDmAMA2JCsXV1VTVHSOQEHhO6cWUsvBBvKtvCiwSfZmHlKV8ceLPryhhmkpJRU5TiaCrLImpBwYAEkPA9ZQtiJED2Jsx4Med8zRMffnRDR5H/vbcW49GZbp+NQ45I8zQfnne/vyC9x937JamkqGfH9f7+zqQ3c6yEizhwzGrhoPb1v9/t0jbyg+HRDoAAAAASUVORK5CYII="
def _mystery_photo():
    return np.asarray(Image.open(io.BytesIO(base64.b64decode(_MYSTERY))).convert("RGB"), dtype=np.float32) / 255

def mystery_painting():
    """A famous painting, 64 x 64, as (X, Y) training data. No peeking."""
    photo = _mystery_photo()
    return pixel_grid(64), photo.reshape(-1, 3).T.copy()

def _img(y):
    y = np.asarray(y, dtype=np.float32); n = int(round(np.sqrt(y.shape[1])))
    return np.clip(np.nan_to_num(y).T.reshape(n, n, 3), 0, 1)

def _noaxes(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)

def _count(betas, omegas):
    return sum(b.size + O.size for b, O in zip(betas, omegas))

def show_photo(photo, title):
    fig, ax = plt.subplots(figsize=(3.6, 3.6)); ax.imshow(photo); _noaxes(ax); ax.set_title(title, fontsize=10); plt.show()

def show_output(y, title, size=3.6):
    fig, ax = plt.subplots(figsize=(size, size)); ax.imshow(_img(y)); _noaxes(ax); ax.set_title(title, fontsize=10); plt.show()

def show_table(X, Y, idx):
    """A few training examples as rows: input (x1, x2) and output (red, green, blue)."""
    fig, ax = plt.subplots(figsize=(7.4, .55 + .42 * len(idx))); ax.axis("off")
    ax.set_xlim(0, 1); ax.set_ylim(-len(idx) - .5, .6)
    cols = [("example", .0), ("x₁", .16), ("x₂", .3), ("red", .46), ("green", .59), ("blue", .72), ("colour", .86)]
    for name, x in cols: ax.text(x, 0, name, weight="bold", va="center")
    ax.text(.16, .45, "input X", color=GRAY, fontsize=9); ax.text(.46, .45, "output Y", color=GRAY, fontsize=9)
    for r, i in enumerate(idx, 1):
        vals = [f"{i}", f"{X[0, i]:+.2f}", f"{X[1, i]:+.2f}", f"{Y[0, i]:.2f}", f"{Y[1, i]:.2f}", f"{Y[2, i]:.2f}"]
        for (name, x), v in zip(cols, vals): ax.text(x, -r, v, va="center", family="monospace")
        ax.add_patch(Rectangle((.86, -r - .32), .08, .64, color=np.clip(Y[:, i], 0, 1), ec=GRAY, lw=.6))
    plt.show()

def show_data(X, Y, mark=1263):
    """Every training example as a dot at its input position, coloured by its output."""
    fig, ax = plt.subplots(figsize=(5.4, 5.4))
    ax.scatter(X[0], X[1], c=np.clip(Y.T, 0, 1), marker="s", s=34, lw=0)
    ax.scatter(X[0, mark], X[1, mark], s=160, facecolors="none", edgecolors=INK, lw=2)
    ax.annotate(f"example {mark}", (X[0, mark], X[1, mark]), xytext=(-.95, -.9), fontsize=10,
                arrowprops=dict(arrowstyle="->", color=INK))
    ax.set_xlabel("x₁"); ax.set_ylabel("x₂"); ax.set_aspect("equal")
    ax.set_title(f"the training data: {X.shape[1]:,} examples, one dot each"); plt.show()

def try_it(fn):
    """Run fn() and print the error message instead of stopping the notebook."""
    try:
        fn(); print("no error")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

def show_units(hs, layer, n=6):
    """The n most active hidden units of one layer, each drawn as an image over the input plane."""
    h = hs[layer]; px = int(round(np.sqrt(h.shape[1]))); busiest = np.argsort(-h.std(axis=1))[:n]
    fig, ax = plt.subplots(1, n, figsize=(2.3 * n, 2.6))
    for j, a in zip(busiest, ax):
        a.imshow(h[j].reshape(px, px), cmap="magma"); _noaxes(a); a.set_title(f"unit {j}", fontsize=9)
    fig.suptitle(f"layer {layer}: the {n} busiest of its {h.shape[0]} hidden units", x=.02, ha="left", weight="bold")
    fig.tight_layout(); plt.show()

def show_training(run):
    base = float(np.mean((run.y - run.y.mean(1, keepdims=True)) ** 2))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.9, 1]})
    ax[0].semilogy(run.losses, color=BLUE, lw=2.2, label="training loss")
    ax[0].axhline(base, color=GRAY, ls="--", lw=1.2, label="baseline: predict the mean colour")
    ax[0].set_xlabel("step"); ax[0].set_title("loss"); ax[0].legend(frameon=False, fontsize=9)
    ax[1].imshow(_img(forward(run.x, run.betas, run.omegas)[1])); _noaxes(ax[1]); ax[1].set_title("output")
    fig.tight_layout(); plt.show()
    print(f"final loss {run.losses[-1]:.5f}   baseline {base:.5f}")

def replay(*runs, names=None):
    names = names or [f"{len(r.widths) - 2} hidden layers" for r in runs]
    n = min(len(r.snapshots) for r in runs)
    fig, ax = plt.subplots(1, len(runs), figsize=(4 * len(runs), 4.4), squeeze=False); ax = ax[0]
    ims = [a.imshow(_img(r.snapshots[0])) for a, r in zip(ax, runs)]
    for a in ax: _noaxes(a)
    def frame(i):
        for a, im, r, name in zip(ax, ims, runs, names):
            im.set_data(_img(r.snapshots[i])); a.set_title(f"{name} · step {40 * i}", fontsize=11)
        return ims
    anim = animation.FuncAnimation(fig, frame, frames=n, interval=100)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode="once"))

def show_errors(run):
    y_pred = forward(run.x, run.betas, run.omegas)[1]
    err = np.abs(y_pred - run.y).mean(0) * 255; n = int(round(np.sqrt(err.size)))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
    ax[0].imshow(_img(y_pred)); ax[0].set_title("output")
    m = ax[1].imshow(err.reshape(n, n), cmap="magma"); ax[1].set_title("error per pixel (colour levels)")
    fig.colorbar(m, ax=ax[1], fraction=.046)
    for a in ax: _noaxes(a)
    fig.tight_layout(); plt.show()

def compare(shallow, deep):
    """The target, then both networks drawn at 400 x 400."""
    grid = pixel_grid(400)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5.4))
    ax[0].imshow(_img(shallow.y)); ax[0].set_title("target (48 × 48)", fontsize=11)
    for a, r, name in ((ax[1], shallow, "shallow"), (ax[2], deep, "deep")):
        a.imshow(_img(forward(grid, r.betas, r.omegas)[1]))
        a.set_title(f"{name}: {len(r.widths) - 2} hidden layer{'s' * (len(r.widths) > 3)}, "
                    f"{_count(r.betas, r.omegas):,} parameters", fontsize=11)
        a.set_xlabel(f"final loss {r.losses[-1]:.5f}", fontsize=11)
    for a in ax: _noaxes(a)
    fig.tight_layout(); plt.show()

def show_overfit(run, used):
    """The training pixels it saw, and what it made of the rest."""
    full = forward(X, run.betas, run.omegas)[1]
    test = float(np.mean((full[:, ~used] - Y[:, ~used]) ** 2))
    seen = np.ones_like(Y); seen[:, used] = Y[:, used]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5.3))
    ax[0].imshow(_img(seen)); ax[0].set_title(f"the {used.sum()} pixels it trained on", fontsize=11)
    ax[1].imshow(_img(full)); ax[1].set_title(f"its guess: train loss {run.losses[-1]:.5f}", fontsize=11)
    ax[2].imshow(_img(Y)); ax[2].set_title(f"the truth: test loss {test:.5f}", fontsize=11)
    for a in ax: _noaxes(a)
    fig.tight_layout(); plt.show()

_REFERENCE = {}
def report(run, title):
    """Loss vs a reference run, gradient size per layer at step 0 (from YOUR backward), and the output."""
    key = tuple(run.widths)
    if key not in _REFERENCE:
        _REFERENCE[key] = train(run.widths)
    ref = _REFERENCE[key]
    def grads0(widths, scale, x, y):
        b, o = init_params(widths, scale); hs, y_pred = forward(x, b, o)
        return [float(np.linalg.norm(d)) for d in backward(hs, y_pred, y, o)[1]]
    g, g_ref = grads0(run.widths, run.scale, run.x, run.y), grads0(ref.widths, "he", ref.x, ref.y)
    base = float(np.mean((run.y - run.y.mean(1, keepdims=True)) ** 2))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1.7, 1.2, 1]})
    ax[0].semilogy(ref.losses, color=FAINT, lw=2.4, label="reference run")
    ax[0].semilogy(run.losses, color=BLUE, lw=2.2, marker="o" if len(run.losses) < 10 else None, label="this run")
    ax[0].axhline(base, color=GRAY, ls="--", lw=1.2, label="baseline: predict the mean colour")
    top = max(30.0, float(np.nanmax(run.losses)) * 30)
    ax[0].set_ylim(1e-4, min(top, 1e38)); ax[0].set_xlabel("step"); ax[0].set_title("loss")
    ax[0].legend(frameon=False, fontsize=9, loc="lower left")
    i = np.arange(len(g)); gv = np.clip(np.nan_to_num(g, nan=1e38, posinf=1e38), 1e-15, 1e38)
    ax[1].bar(i, gv, .6, color=BLUE, label="this run")
    ax[1].scatter(i, g_ref, marker="_", s=160, lw=2.4, color=INK, zorder=3, label="reference run")
    ax[1].set_yscale("log"); ax[1].set_ylim(min(gv.min(), min(g_ref)) / 5, max(gv.max(), max(g_ref)) * 5)
    ax[1].set_xticks(i, [f"Ω{k}".translate(_SUB) if len(i) < 12 or k % 4 == 0 or k == len(i) - 1 else "" for k in i])
    ax[1].set_title("gradient size per layer at step 0,  ‖∂L/∂Ω‖", fontsize=10.5)
    ax[1].legend(frameon=False, fontsize=9); ax[1].grid(axis="x", visible=False)
    ax[2].imshow(_img(forward(run.x, run.betas, run.omegas)[1])); _noaxes(ax[2]); ax[2].set_title("output")
    fig.suptitle(title, x=.07, ha="left", fontsize=15, weight="bold"); fig.tight_layout(); plt.show()
    end = run.losses[-1]
    ok = np.isfinite(end) and end < 3 * ref.losses[-1]
    print(f"final loss {end:.4g}   reference {ref.losses[-1]:.4g}   baseline {base:.4f}   ->  {'trains' if ok else 'FAILS'}")

def reveal(run):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5.4))
    ax[0].imshow(_mystery_photo()); ax[0].set_title("Edvard Munch, The Scream (1893)", fontsize=11)
    ax[1].imshow(_img(forward(run.x, run.betas, run.omegas)[1])); ax[1].set_title("your network, 64 × 64", fontsize=11)
    ax[2].imshow(_img(forward(pixel_grid(400), run.betas, run.omegas)[1])); ax[2].set_title("your network, 400 × 400", fontsize=11)
    for a in ax: _noaxes(a)
    fig.tight_layout(); plt.show()

def score(run, budget=25000):
    p = _count(run.betas, run.omegas)
    print(f"{run.widths}, lr = {run.lr}")
    print(f"parameters: {p:,}  {'(within budget)' if p <= budget else '(OVER the ' + format(budget, ',') + ' budget)'}")
    print(f"final loss: {run.losses[-1]:.5f}")

print("helpers ready")

---
## 1 · The data  <small>0–9 min</small>

▶ Load the image and check it: shape, type, range.

In [ ]:
photo = robbys_cat()
print(photo.shape, photo.dtype, photo.min(), photo.max())
show_photo(photo, "Robby's cat, 48 × 48. The coffee is Robby's.")

A network trains on a **table**: one input and one output per example. Here every pixel is an example:

$$\text{input } x = (x_1, x_2) \text{ = its position} \qquad \text{output } y = (\text{red}, \text{green}, \text{blue}) \text{ = its colour}$$

▶ Build the inputs: every pixel's position, scaled to −1…1, one column per pixel.

In [ ]:
# coords = np.linspace(-1, 1, 48, dtype=np.float32)
coords = ______
# x1, x2 = np.meshgrid(coords, -coords)
x1, x2 = ______
# X = np.vstack([x1.ravel(), x2.ravel()])
X = ______
print("X:", X.shape)

▶ Build the outputs: every pixel's colour, one column per pixel.

In [ ]:
# Y = photo.reshape(-1, 3).T
Y = ______
print("Y:", Y.shape)

▶ Look at a few rows of the table.

In [ ]:
show_table(X, Y, [0, 47, 1263, 2303])

**Question for class:** Which part of the cat is example 1263?

▶ Look at the whole training set.

In [ ]:
show_data(X, Y)

**A silent data bug.**

▶ Reshape the colours the "obvious" way and draw them.

In [ ]:
# Y_wrong = photo.reshape(3, -1)
Y_wrong = ______
print("Y_wrong:", Y_wrong.shape)
show_output(Y_wrong, "Y_wrong, drawn")

**Question for class:** Right shape, no error. What went wrong?

**Takeaway:** the colours are stored red, green, blue, red, green, blue…; `reshape(3, -1)` deals them into
the wrong rows. A network would train on this without complaint. **Draw your data after every reshape.**

| data check | ours |
|---|---|
| one column per example | `X` is 2 × 2304, `Y` is 3 × 2304 |
| inputs on a small, centred scale | −1 … 1 |
| outputs in a known range | 0 … 1 |
| looks right when drawn | ✓ |

---
## 2 · The network  <small>9–18 min · slides 43, 45–52</small>

$$h_1 = a[\beta_0 + \Omega_0 x] \qquad h_{k+1} = a[\beta_k + \Omega_k h_k] \qquad y = \beta_K + \Omega_K h_K$$

Layer $k$ has a weight matrix $\Omega_k$ ($D_{out} \times D_{in}$) and a bias $\beta_k$ ($D_{out} \times 1$).

▶ The activation function.

In [ ]:
def relu(z):
    # return np.maximum(0, z)
    return ______

▶ Create the parameters. **He initialization** (ch. 7): random $\Omega$ with spread $\sqrt{2/D_{in}}$, $\beta = 0$.

In [ ]:
def init_params(widths, scale="he", seed=0):
    rng = np.random.default_rng(seed)
    betas, omegas = [], []
    for d_in, d_out in zip(widths[:-1], widths[1:]):
        # sd = np.sqrt(2 / d_in) if scale == "he" else scale
        sd = ______
        # omegas.append(sd * rng.standard_normal((d_out, d_in), dtype=np.float32))
        ______
        # betas.append(np.zeros((d_out, 1), dtype=np.float32))
        ______
    omegas[-1] *= 0.1          # start the output layer small...
    betas[-1] += 0.5           # ...so the first guess is mid-gray
    return betas, omegas

▶ Pick a shape: 2 inputs, five hidden layers of 40, 3 outputs. Print every parameter's shape.

In [ ]:
# DEEP = [2, 40, 40, 40, 40, 40, 3]
DEEP = ______
betas, omegas = init_params(DEEP)
for k, (b, O) in enumerate(zip(betas, omegas)):
    print(f"layer {k}:   Ω{k} {O.shape}   β{k} {b.shape}")
# n_params = sum(b.size + O.size for b, O in zip(betas, omegas))
n_params = ______
print("parameters:", n_params)

**Question for class:** 6,803 parameters, 2,304 training examples. Is that a problem? *(Hold that thought until section 5.)*

▶ The forward pass (slide 52). Keep every layer: backprop needs them, and so do we.

In [ ]:
def forward(x, betas, omegas):
    hs = [x]
    for k in range(len(betas) - 1):
        # hs.append(relu(betas[k] + omegas[k] @ hs[-1]))
        ______
    # y_pred = betas[-1] + omegas[-1] @ hs[-1]
    y_pred = ______
    return hs, y_pred

▶ Run it, untrained.

In [ ]:
hs, y_pred = forward(X, betas, omegas)
print("hidden layers:", [h.shape for h in hs[1:]], "   output:", y_pred.shape)
show_output(y_pred, "before training")

**Normal errors.** You will see both of these many times this semester.

▶ Error 1: pass the data with rows and columns swapped.

In [ ]:
try_it(lambda: forward(X.T, betas, omegas))

**Question for class:** Which two numbers does the message say disagree, and where do they come from?

▶ Error 2: a bias vector shaped (40,) instead of (40, 1).

In [ ]:
flat_betas = [b.ravel() for b in betas]
try_it(lambda: forward(X, flat_betas, omegas))

**Takeaway:** shape errors name the two shapes that disagree. Print the shapes, then fix the one that is wrong.

---
## 3 · Loss and backpropagation  <small>18–26 min · ch. 5 and 7 preview</small>

**Loss:** the mean squared error over every pixel and colour channel,
$L = \text{mean}\big[(y_{pred} - y)^2\big]$. A useful baseline: paint every pixel the mean colour.

▶ Write the loss and measure the untrained network against the baseline.

In [ ]:
def loss(y_pred, y):
    # return np.mean((y_pred - y) ** 2)
    return ______

# baseline = loss(Y.mean(axis=1, keepdims=True), Y)
baseline = ______
print(f"untrained: {loss(y_pred, Y):.4f}    baseline (mean colour): {baseline:.4f}")

**Backpropagation:** start with the error at the output, then walk backwards one layer at a time.
With $g = \partial L / \partial(\text{layer } k\text{'s output})$:

$$\frac{\partial L}{\partial \Omega_k} = g\,h_k^{T} \qquad \frac{\partial L}{\partial \beta_k} = \textstyle\sum g \qquad g \leftarrow \big(\Omega_k^{T} g\big) \odot \big[h_k > 0\big]$$

The last step passes the error back through $\Omega_k$, then through the ReLU, which lets it through only where the unit was on.

▶ Write it.

In [ ]:
def backward(hs, y_pred, y, omegas):
    # g = 2 * (y_pred - y) / y.size
    g = ______
    dbetas, domegas = [], []
    for k in reversed(range(len(omegas))):
        # domegas.insert(0, g @ hs[k].T)
        ______
        # dbetas.insert(0, g.sum(axis=1, keepdims=True))
        ______
        # g = (omegas[k].T @ g) * (hs[k] > 0)
        g = ______
    return dbetas, domegas

▶ Check it: nudge one weight up and down, and see how the loss moves (a **gradient check**).

In [ ]:
dbetas, domegas = backward(hs, y_pred, Y, omegas)
eps = 1e-3
omegas[1][0, 0] += eps;      up = loss(forward(X, betas, omegas)[1], Y)
omegas[1][0, 0] -= 2 * eps;  down = loss(forward(X, betas, omegas)[1], Y)
omegas[1][0, 0] += eps
# numeric = (up - down) / (2 * eps)
numeric = ______
print(f"backprop says {domegas[1][0, 0]:+.6f}    nudging says {numeric:+.6f}")

**Takeaway:** one backward pass gives all 6,803 gradients at once: how much the loss changes if each
weight moves a little. When you write your own gradients, always check them this way.

---
## 4 · Training, and reading the result  <small>26–36 min</small>

Every step: **predict → measure → backprop → update.** Plain gradient descent (IC 2):
$\theta \leftarrow \theta - \alpha\, \partial L / \partial \theta$.

▶ Write the training loop.

In [ ]:
def train_gd(widths, lr, steps=1000):
    betas, omegas = init_params(widths)
    for step in range(steps + 1):
        # hs, y_pred = forward(X, betas, omegas)
        hs, y_pred = ______
        L = loss(y_pred, Y)
        if step % 250 == 0 or not np.isfinite(L):
            print(f"step {step:4d}   loss {L:.4f}")
        if not np.isfinite(L):
            break
        # dbetas, domegas = backward(hs, y_pred, Y, omegas)
        dbetas, domegas = ______
        for k in range(len(betas)):
            # betas[k] -= lr * dbetas[k]
            ______
            # omegas[k] -= lr * domegas[k]
            ______
    return betas, omegas

▶ Train with learning rate 0.3.

In [ ]:
betas, omegas = train_gd(DEEP, lr=0.3)
show_output(forward(X, betas, omegas)[1], "gradient descent, 1,000 steps")

▶ Error 3: the same loop with learning rate 1.0.

In [ ]:
train_gd(DEEP, lr=1.0);

**Question for class:** The loss rose, then became `nan`. What happened to the numbers?

**Takeaway:** `nan` (*not a number*) means values overflowed. When the loss blows up, suspect the learning rate first.

---

Plain gradient descent works, but it is slow: one step size for 6,803 very different parameters.
**Adam** (ch. 6) keeps a running average of each gradient ($m$) and of its square ($v$), and steps by
$m/\sqrt{v}$, so every parameter gets a sensibly sized step:

$$m \leftarrow 0.9\,m + 0.1\,g \qquad v \leftarrow 0.999\,v + 0.001\,g^2 \qquad \theta \leftarrow \theta - \alpha\,\frac{\hat m}{\sqrt{\hat v} + 10^{-8}}$$

($\hat m$, $\hat v$ correct the running averages for starting at zero.)

▶ The same loop with Adam, keeping a record of the loss. Fill in Adam's three lines.

In [ ]:
def train(widths, lr=0.005, steps=2000, scale="he", data=None):
    x, y = (X, Y) if data is None else data
    betas, omegas = init_params(widths, scale)
    params = betas + omegas
    m = [np.zeros_like(p) for p in params]
    v = [np.zeros_like(p) for p in params]
    losses, snapshots = [], []
    for t in range(1, steps + 1):
        hs, y_pred = forward(x, betas, omegas)
        losses.append(loss(y_pred, y))
        if not np.isfinite(losses[-1]):
            break
        dbetas, domegas = backward(hs, y_pred, y, omegas)
        for p, g, m_, v_ in zip(params, dbetas + domegas, m, v):
            # m_[:] = 0.9 * m_ + 0.1 * g
            m_[:] = ______
            # v_[:] = 0.999 * v_ + 0.001 * g ** 2
            v_[:] = ______
            # p -= lr * (m_ / (1 - 0.9 ** t)) / (np.sqrt(v_ / (1 - 0.999 ** t)) + 1e-8)
            ______
        if t % 40 == 1:
            snapshots.append(y_pred)
    return SimpleNamespace(widths=widths, scale=scale, lr=lr, betas=betas, omegas=omegas,
                           losses=losses, snapshots=snapshots, x=x, y=y)

▶ Train with Adam.

In [ ]:
# deep = train(DEEP)
deep = ______
show_training(deep)

**Reading the result.** A loss of 0.0006 means nothing on its own. The loss is a mean *squared* error on
colours from 0 to 1, so its square root, times 255, is the typical error in colour levels.

▶ Translate the loss.

In [ ]:
# print(f"typical error: {np.sqrt(deep.losses[-1]) * 255:.1f} colour levels out of 255")
______
print(f"the baseline:  {np.sqrt(baseline) * 255:.1f} colour levels out of 255")

▶ Watch it learn.

In [ ]:
replay(deep)

▶ Where is it still wrong?

In [ ]:
show_errors(deep)

**Question for class:** Where are the errors, and why there?

**Takeaway:** the errors sit on edges and small details (pupils, whiskers, the mug's rim), where the colour
changes fastest. Watch **where** a model fails, not just its average loss.

▶ Look inside: run the trained network on a 200 × 200 grid and draw individual hidden units in layers 1, 3 and 5.

In [ ]:
# hs_grid, _ = forward(pixel_grid(200), deep.betas, deep.omegas)
hs_grid, _ = ______
show_units(hs_grid, layer=1)
show_units(hs_grid, layer=3)
show_units(hs_grid, layer=5)

**Question for class:** Layer-1 units are straight ramps (IC 3). What are layer-5 units responding to?

**Takeaway:** each layer builds on the one before (slides 27–29): straight edges → blobs → parts of the cat
(the silhouette, the eyes, the mug). Nobody told it what an eye is; the loss did.

▶ Last week's network, one hidden layer, the same number of parameters. Both get 1,000 steps. *(the shallow one takes ~10 s)*

In [ ]:
SHALLOW = [2, 1133, 3]
# shallow = train(SHALLOW, lr=0.01, steps=1000)
shallow = ______
# deep_1k = train(DEEP, steps=1000)
deep_1k = ______
compare(shallow, deep_1k)

**Question for class:** Same parameters, same steps. Why is deep so much better?

**Takeaway:** each shallow unit is one straight cut across the whole image; a deep network folds and re-cuts,
so it gets far more pieces per parameter and puts them where the detail is (slides 57–61). It also
trains faster (slide 64).

---
## 5 · Does it generalize?  <small>36–44 min · ch. 8 preview</small>

A low training loss can mean it memorized the examples. The real test: pixels it has **never** seen.

▶ Hold out 20% of the pixels as a test set.

In [ ]:
rng = np.random.default_rng(1)
# is_test = rng.random(X.shape[1]) < 0.2
is_test = ______
X_train, Y_train = X[:, ~is_test], Y[:, ~is_test]
X_test, Y_test = X[:, is_test], Y[:, is_test]
print("train:", X_train.shape[1], "   test:", X_test.shape[1])

▶ Train on the training pixels only, then score both sets.

In [ ]:
# split = train(DEEP, data=(X_train, Y_train))
split = ______
# test_loss = loss(forward(X_test, split.betas, split.omegas)[1], Y_test)
test_loss = ______
print(f"train loss {split.losses[-1]:.5f}  ({np.sqrt(split.losses[-1]) * 255:.1f} levels)")
print(f"test loss  {test_loss:.5f}  ({np.sqrt(test_loss) * 255:.1f} levels)")

**Question for class:** The test loss is several times the training loss. Is the model broken?

**Takeaway:** some gap is normal: it has never seen those pixels. Judge the gap in real units
(colour levels), and compare it with the baseline.

▶ Now starve it: train on 3% of the pixels.

In [ ]:
# few = rng.random(X.shape[1]) < 0.03
few = ______
tiny = train(DEEP, data=(X[:, few], Y[:, few]))
show_overfit(tiny, few)

**Question for class:** Training loss is almost zero. Did it learn the cat?

**Takeaway:** **overfitting.** 6,803 parameters and 68 examples: it fits every training pixel and guesses
everything in between. Near-zero training loss plus a large test loss means you need more data (or a smaller
model). With 2,304 examples, the same network was fine.

▶ The biggest test of all: every position **between** the pixels. Draw the trained network on a 400 × 400 grid.

In [ ]:
# grid400 = pixel_grid(400)
grid400 = ______
# enhanced = forward(grid400, deep.betas, deep.omegas)[1]
enhanced = ______
show_output(enhanced, "trained on 48 × 48, drawn at 400 × 400", size=6)

▶ More capacity: six hidden layers of 64.

In [ ]:
# BIGGER = [2, 64, 64, 64, 64, 64, 64, 3]
BIGGER = ______
bigger = train(BIGGER, lr=0.003)
show_output(forward(grid400, bigger.betas, bigger.omegas)[1], "21,187 parameters, drawn at 400 × 400", size=6)

**Takeaway:** trained on 2,304 points, it draws a smooth, sharp cat at any resolution, because it learned a
function of position rather than a list of pixels.

---
## 6 · Common pitfalls  <small>44–54 min · slide 63</small>

Each run has **one** thing wrong. Read the three plots, diagnose, fix one argument.
Gray line and black marks: a **reference run** with nothing wrong. The middle plot comes from **your** `backward`.

### Run A
▶ Train and plot.

In [ ]:
run_a = train(DEEP, lr=0.00001)
report(run_a, "Run A")

**Question for class:** The gradients match the reference. Why is the loss so slow?

▶ Fix it.

In [ ]:
# fixed = train(DEEP, lr=0.005)
fixed = ______
report(fixed, "Run A, fixed")

**Takeaway:** **learning rate too small**: tiny steps, training stalls.

### Run B
▶ Train and plot.

In [ ]:
run_b = train(DEEP, lr=1.0)
report(run_b, "Run B")

**Question for class:** The loss shot up, then flattened exactly on the baseline. What happened to the network?

▶ Fix it.

In [ ]:
# fixed = train(DEEP, lr=0.005)
fixed = ______
report(fixed, "Run B, fixed")

**Takeaway:** **learning rate too large**: the first steps blew the weights up, every ReLU switched off,
and the network can only output a constant: the mean colour.

---

Runs C and D use a much deeper network: 16 hidden layers.

▶ Define it.

In [ ]:
VERY_DEEP = [2] + [32] * 16 + [3]

### Run C
▶ Train and plot. This run starts every weight with spread 0.05.

In [ ]:
run_c = train(VERY_DEEP, scale=0.05)
report(run_c, "Run C")

**Question for class:** Read the gradient bars from right to left. What happens on the way back to $\Omega_0$?

▶ Fix it.

In [ ]:
# fixed = train(VERY_DEEP, scale="he")
fixed = ______
report(fixed, "Run C, fixed")

**Takeaway:** **vanishing gradients.** Backprop multiplies by $\Omega_k^T$ once per layer; small weights
shrink the signal at every layer, and after 16 layers nothing reaches $\Omega_0$.

### Run D
▶ Train and plot. This run starts every weight with spread 1.0.

In [ ]:
run_d = train(VERY_DEEP, scale=1.0)
report(run_d, "Run D")

**Question for class:** How large are the gradients compared with the reference?

▶ Fix it.

In [ ]:
# fixed = train(VERY_DEEP, scale="he")
fixed = ______
report(fixed, "Run D, fixed")

**Takeaway:** **exploding gradients**: big weights grow the signal at every layer. He initialization,
spread $\sqrt{2/D_{in}}$, is the one scale that keeps every layer the same size.

### Run E
▶ Train and plot. This run's inputs are pixel numbers, 0 to 47, instead of −1 to 1.

In [ ]:
X_pixels = X * 23.5 + 23.5
run_e = train(DEEP, data=(X_pixels, Y))
report(run_e, "Run E")

**Question for class:** No crash and no NaN. What do the loss and the gradient bars tell you?

▶ Fix it.

In [ ]:
# fixed = train(DEEP, data=(X, Y))
fixed = ______
report(fixed, "Run E, fixed")

**Takeaway:** **unscaled inputs**: inputs 25× too big make every layer's signal and gradient too big. It still
trains, just badly (about 20× worse). Not every bug crashes.
Scale inputs to a small, centred range.

| run | what the plots show | cause | fix |
|---|---|---|---|
| A | loss falls very slowly; gradients normal | learning rate too small | raise it |
| B | loss spikes, then sits on the baseline | learning rate too large | lower it |
| C | gradients shrink toward $\Omega_0$ | vanishing gradients (weights too small) | He initialization |
| D | gradients and loss astronomically large | exploding gradients (weights too large) | He initialization |
| E | trains, but far worse; gradients ~100× too big in every layer | unscaled inputs | scale to −1 … 1 |

---
## 7 · The mystery painting  <small>54–60 min</small>

A famous painting, 64 × 64, loaded as training data only. Nobody has seen it.

▶ Load it. No peeking.

In [ ]:
MX, MY = mystery_painting()
print("X:", MX.shape, "   Y:", MY.shape)

▶ Train on it and watch. Call out the painting as soon as you recognize it.

In [ ]:
# guess = train(DEEP, data=(MX, MY))
guess = ______
replay(guess)

▶ Reveal.

In [ ]:
reveal(guess)

**Class challenge:** lowest final loss on the painting wins. At most **25,000** parameters.
Change only the widths and the learning rate.

▶ Your entry.

In [ ]:
my_widths = [2, 40, 40, 40, 40, 40, 3]
my_lr = 0.005

entry = train(my_widths, lr=my_lr, data=(MX, MY))
score(entry)

▶ Your entry, revealed.

In [ ]:
reveal(entry)

**What you built:** data → `init_params` → `forward` → `loss` → `backward` → Adam, in about 40 lines of numpy.
Every deep learning library does these same steps. **Next:** loss functions (ch. 5), optimizers (ch. 6),
and the full story of backprop and initialization (ch. 7).